# Project 4 — Synthetic Cybersecurity Threat Detection with Spark MLlib

**Educational demonstration only.** This notebook generates synthetic network events, trains and compares two Spark MLlib classifiers, logs MLflow metrics, computes feature importance, and exports a technical report. Synthetic labels are assigned before feature generation, making the classes artificially easy to distinguish. **Do not interpret the test accuracy as real-world intrusion-detection performance.**

**Runtime:** Databricks Free Edition Serverless, Spark 4.2.0, Python; MLflow 3.12.0 in the verified run.

**Execution:** Run cells top-to-bottom in a Databricks workspace with a Unity Catalog volume at `/Volumes/workspace/usda_training/raw_files/`. If your catalog/schema/volume differs, update those paths. The CSV generator preserves an existing file; if it already exists, ensure it matches this project before continuing. MLflow cells create fresh runs whenever executed.

**Provenance:** Reorganized from an exported Databricks notebook. One missing CSV-load cell was restored, and feature validation was moved after feature engineering. Outputs were cleared for publication. The corrected export has **not been independently reexecuted**; see the companion report for results observed in the original session.


## 1. Environment and synthetic data


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("PROJECT 4: CYBERSECURITY THREAT DETECTION")
print("---------------------------------------")

print("Spark version:", spark.version)

spark.sql("SHOW SCHEMAS IN workspace").show()

spark.sql(
    "SHOW VOLUMES IN workspace.usda_training"
).show()

print("---------------------------------------")
print("PROJECT 4 ENVIRONMENT READY")


In [ ]:
from pathlib import Path

project_base = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "cyber_threat_detection"
)

directories = [
    project_base / "data",
    project_base / "output",
    project_base / "reports"
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print("Ready:", directory)

print("PROJECT 4 DIRECTORIES READY")


In [ ]:
import csv
import random
from pathlib import Path
from datetime import datetime, timedelta

random.seed(42)

project_base = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "cyber_threat_detection"
)

output_file = project_base / "data" / "network_events.csv"

records = []
start_time = datetime(2026, 1, 1)

for i in range(5000):
    event_time = start_time + timedelta(
        minutes=i * 5
    )

    # Synthetic threat classification
    is_suspicious = int(random.random() < 0.15)

    if is_suspicious:
        duration = random.randint(1, 45)
        failed_logins = random.randint(3, 15)
        bytes_sent = random.randint(5000, 80000)
        bytes_received = random.randint(100, 10000)
        destination_port = random.choice(
            [21, 22, 23, 3389, 445]
        )
    else:
        duration = random.randint(20, 600)
        failed_logins = random.randint(0, 2)
        bytes_sent = random.randint(1000, 100000)
        bytes_received = random.randint(5000, 150000)
        destination_port = random.choice(
            [80, 443, 53, 25, 110, 143]
        )

    records.append([
        f"EVT{i+1:06d}",
        event_time.isoformat(),
        duration,
        failed_logins,
        bytes_sent,
        bytes_received,
        destination_port,
        is_suspicious
    ])

if output_file.exists():
    print("Existing dataset preserved:", output_file)
else:
    with output_file.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "event_id",
            "event_timestamp",
            "duration_seconds",
            "failed_login_attempts",
            "bytes_sent",
            "bytes_received",
            "destination_port",
            "is_suspicious"
        ])
        writer.writerows(records)

    print("Synthetic network dataset created.")

print("File:", output_file)
print("Rows generated:", len(records))
print("DATASET GENERATION COMPLETED")


## 2. Load and validate the CSV


In [ ]:
from pyspark.sql import functions as F, types as T

data_path = (
    "/Volumes/workspace/usda_training/raw_files/"
    "cyber_threat_detection/data/network_events.csv"
)

schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("event_timestamp", T.StringType()),
    T.StructField("duration_seconds", T.IntegerType()),
    T.StructField("failed_login_attempts", T.IntegerType()),
    T.StructField("bytes_sent", T.LongType()),
    T.StructField("bytes_received", T.LongType()),
    T.StructField("destination_port", T.IntegerType()),
    T.StructField("is_suspicious", T.IntegerType()),
])

security_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema)
    .load(data_path)
)
print("Total network events:", security_df.count())
security_df.printSchema()
print("DATASET LOAD COMPLETED")


In [ ]:
from pyspark.sql import functions as F

total = security_df.count()

label_counts = (
    security_df
    .groupBy("is_suspicious")
    .count()
    .orderBy("is_suspicious")
)

print("SECURITY EVENT DISTRIBUTION")
print("---------------------------")

label_counts.show()

duplicate_ids = (
    security_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_labels = security_df.filter(
    F.col("is_suspicious").isNull()
).count()

print("Total events:", total)
print("Duplicate event IDs:", duplicate_ids)
print("Missing labels:", missing_labels)

assert total == 5000
assert duplicate_ids == 0
assert missing_labels == 0

print("DATA QUALITY VALIDATION PASSED")


## 3. Explore event patterns


In [ ]:
from pyspark.sql import functions as F

print("NETWORK SECURITY EXPLORATORY ANALYSIS")
print("-------------------------------------")

traffic_summary = (
    security_df
    .groupBy("is_suspicious")
    .agg(
        F.count("*").alias("event_count"),
        F.round(
            F.avg("duration_seconds"), 2
        ).alias("avg_duration_seconds"),
        F.round(
            F.avg("failed_login_attempts"), 2
        ).alias("avg_failed_logins"),
        F.round(
            F.avg("bytes_sent"), 2
        ).alias("avg_bytes_sent"),
        F.round(
            F.avg("bytes_received"), 2
        ).alias("avg_bytes_received")
    )
    .orderBy("is_suspicious")
)

display(traffic_summary)

print("EXPLORATORY ANALYSIS COMPLETED")


In [ ]:
port_analysis = (
    security_df
    .groupBy(
        "destination_port",
        "is_suspicious"
    )
    .agg(
        F.count("*").alias("event_count")
    )
    .orderBy(
        "destination_port",
        "is_suspicious"
    )
)

print("DESTINATION PORT ANALYSIS")
print("-------------------------")

display(port_analysis)

print("PORT ANALYSIS COMPLETED")


## 4. Engineer and validate features


In [ ]:
from pyspark.sql import functions as F

feature_df = (
    security_df
    .withColumn(
        "event_timestamp_parsed",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "total_bytes",
        F.col("bytes_sent") +
        F.col("bytes_received")
    )
    .withColumn(
        "send_receive_ratio",
        F.col("bytes_sent") /
        F.greatest(
            F.col("bytes_received"),
            F.lit(1)
        )
    )
    .withColumn(
        "failed_login_rate",
        F.col("failed_login_attempts") /
        F.greatest(
            F.col("duration_seconds"),
            F.lit(1)
        )
    )
)

print("ENGINEERED SECURITY FEATURES")
print("----------------------------")

display(
    feature_df.select(
        "event_id",
        "duration_seconds",
        "failed_login_attempts",
        "total_bytes",
        "send_receive_ratio",
        "failed_login_rate",
        "is_suspicious"
    ).limit(10)
)

print("FEATURE ENGINEERING COMPLETED")


In [ ]:
from pyspark.sql import functions as F
from functools import reduce

feature_columns = [
    "duration_seconds",
    "failed_login_attempts",
    "bytes_sent",
    "bytes_received",
    "destination_port",
    "total_bytes",
    "send_receive_ratio",
    "failed_login_rate"
]

# Check for null and NaN values
invalid_conditions = [
    F.col(c).isNull() |
    F.isnan(F.col(c).cast("double"))
    for c in feature_columns
]

# Use Python reduce to combine the conditions
combined_condition = reduce(
    lambda a, b: a | b,
    invalid_conditions
)

invalid_features = feature_df.filter(
    combined_condition
).count()

total_records = feature_df.count()

print("FEATURE VALIDATION RESULTS")
print("--------------------------")
print("Total events:", total_records)
print("Invalid feature records:", invalid_features)

assert total_records == 5000
assert invalid_features == 0

print("FEATURE VALIDATION PASSED")


## 5. Assemble ML features and hold out test records


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_columns = [
    "duration_seconds",
    "failed_login_attempts",
    "bytes_sent",
    "bytes_received",
    "destination_port",
    "total_bytes",
    "send_receive_ratio",
    "failed_login_rate"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="raw_features",
    handleInvalid="error"
)

assembled_df = assembler.transform(feature_df)

ml_df = assembled_df.withColumn(
    "label",
    assembled_df["is_suspicious"].cast("double")
)

print("ML FEATURE PREPARATION")
print("----------------------")
print("Total ML records:", ml_df.count())
print("Number of features:", len(feature_columns))

assert ml_df.count() == 5000

print("ML FEATURE PREPARATION COMPLETED")


In [ ]:
train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Materialize both splits so their contents remain
# stable across subsequent model-training operations.
train_df = train_df.cache()
test_df = test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print("TRAIN / TEST SPLIT")
print("------------------")
print("Training records:", train_count)
print("Testing records:", test_count)

assert train_count + test_count == 5000
assert train_count > 0
assert test_count > 0

print("TRAIN / TEST SPLIT COMPLETED")


**Evaluation caveat:** A seeded random 80/20 split checks this synthetic sample only. For a realistic security assessment, use temporally separated and independently collected labeled events, with attention to class imbalance and distribution shift.


## 6. Train the classifiers


In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(train_df)

train_scaled = scaler_model.transform(train_df)
test_scaled = scaler_model.transform(test_df)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_model = logistic_regression.fit(train_scaled)

lr_predictions = lr_model.transform(test_scaled)

print("LOGISTIC REGRESSION TRAINING COMPLETED")


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

random_forest = RandomForestClassifier(
    featuresCol="raw_features",
    labelCol="label",
    numTrees=100,
    maxDepth=6,
    seed=42
)

rf_model = random_forest.fit(train_df)

rf_predictions = rf_model.transform(test_df)

print("RANDOM FOREST TRAINING COMPLETED")


## 7. Evaluate the held-out test set


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions, model_name):
    results = {}

    for metric in [
        "accuracy",
        "weightedPrecision",
        "weightedRecall",
        "f1"
    ]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        results[metric] = evaluator.evaluate(predictions)

    print(f"\n{model_name}")
    print("-" * 35)

    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")

    confusion = (
        predictions
        .groupBy("label", "prediction")
        .count()
        .orderBy("label", "prediction")
    )

    confusion.show()

    return results

lr_results = evaluate_model(
    lr_predictions,
    "LOGISTIC REGRESSION"
)

rf_results = evaluate_model(
    rf_predictions,
    "RANDOM FOREST"
)

print("MODEL EVALUATION COMPLETED")


In [ ]:
from pyspark.sql import functions as F

def security_metrics(predictions, model_name):
    counts = {
        (int(r["label"]), int(r["prediction"])): r["count"]
        for r in predictions.groupBy("label", "prediction")
            .count().collect()
    }

    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    tp = counts.get((1, 1), 0)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    false_positive_rate = (
        fp / (fp + tn) if fp + tn else 0
    )

    false_negative_rate = (
        fn / (fn + tp) if fn + tp else 0
    )

    result = {
        "model": model_name,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate
    }

    return result

lr_security = security_metrics(
    lr_predictions,
    "Logistic Regression"
)

rf_security = security_metrics(
    rf_predictions,
    "Random Forest"
)

print("SECURITY CLASSIFICATION METRICS")
print("-------------------------------")

for result in [lr_security, rf_security]:
    print(f"\n{result['model']}")
    for key, value in result.items():
        if key == "model":
            continue
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

print("\nSECURITY METRICS COMPLETED")


In [ ]:
comparison_data = [
    lr_security,
    rf_security
]

comparison_df = spark.createDataFrame(
    comparison_data
)

display(
    comparison_df.select(
        "model",
        "TP",
        "TN",
        "FP",
        "FN",
        "precision",
        "recall",
        "f1",
        "false_positive_rate",
        "false_negative_rate"
    )
)

print("MODEL COMPARISON COMPLETED")


In [ ]:
print("MODEL EVALUATION VALIDATION")
print("---------------------------")

assert sum(
    lr_security[k] for k in ["TP", "TN", "FP", "FN"]
) == 958

assert sum(
    rf_security[k] for k in ["TP", "TN", "FP", "FN"]
) == 958

assert lr_security["FN"] == 1
assert lr_security["FP"] == 0

assert rf_security["FN"] == 0
assert rf_security["FP"] == 0

assert rf_security["TP"] == 138
assert rf_security["TN"] == 820

print("All confusion-matrix counts verified.")
print("SECURITY EVALUATION VALIDATION PASSED")


## 8. Log MLflow experiments


In [ ]:
import mlflow

print("MLFLOW ENVIRONMENT")
print("------------------")
print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

# Use the Databricks notebook experiment.
# No separate experiment setup is required.
print("MLFLOW INITIALIZATION COMPLETED")


In [ ]:
import mlflow

model_runs = {}

models_to_log = [
    (
        "Logistic_Regression",
        lr_model,
        lr_security,
        lr_results
    ),
    (
        "Random_Forest",
        rf_model,
        rf_security,
        rf_results
    )
]

for name, model, security, overall in models_to_log:

    with mlflow.start_run(run_name=name) as run:

        mlflow.set_tags({
            "project": "Cybersecurity Threat Detection",
            "data_type": "synthetic",
            "framework": "PySpark MLlib",
            "evaluation": "held-out random test split",
            "limitation": "synthetic class-separable patterns"
        })

        mlflow.log_params({
            "model_type": name,
            "feature_count": len(feature_columns),
            "dataset_size": 5000,
            "training_records": train_count,
            "testing_records": test_count,
            "split_seed": 42
        })

        if name == "Logistic_Regression":
            mlflow.log_params({
                "max_iter": 100,
                "reg_param": 0.01,
                "elastic_net_param": 0.0,
                "feature_scaling": "standard"
            })
        else:
            mlflow.log_params({
                "num_trees": 100,
                "max_depth": 6,
                "random_seed": 42,
                "feature_scaling": "none"
            })

        mlflow.log_metrics({
            "accuracy": float(overall["accuracy"]),
            "weighted_f1": float(overall["f1"]),
            "suspicious_precision": float(security["precision"]),
            "suspicious_recall": float(security["recall"]),
            "suspicious_f1": float(security["f1"]),
            "false_positive_rate": float(
                security["false_positive_rate"]
            ),
            "false_negative_rate": float(
                security["false_negative_rate"]
            ),
            "true_positives": float(security["TP"]),
            "true_negatives": float(security["TN"]),
            "false_positives": float(security["FP"]),
            "false_negatives": float(security["FN"])
        })

        mlflow.log_dict(
            security,
            "reports/security_metrics.json"
        )

        model_runs[name] = run.info.run_id

        print(f"{name} logged successfully.")
        print("Run ID:", run.info.run_id)

print("MLFLOW EXPERIMENT LOGGING COMPLETED")


In [ ]:
import mlflow

print("MLFLOW RUN VERIFICATION")
print("-----------------------")

for name, run_id in model_runs.items():
    run = mlflow.get_run(run_id)

    print("\nModel:", name)
    print("Status:", run.info.status)
    print(
        "Accuracy:",
        round(run.data.metrics["accuracy"], 4)
    )
    print(
        "Suspicious recall:",
        round(run.data.metrics["suspicious_recall"], 4)
    )

    assert run.info.status == "FINISHED"
    assert "suspicious_f1" in run.data.metrics
    assert "feature_count" in run.data.params

print("\nMLFLOW RUN VERIFICATION PASSED")


## 9. Explain the model predictions


In [ ]:
import pandas as pd

rf_importances = rf_model.featureImportances.toArray()

rf_importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_importances
})

rf_importance_df = (
    rf_importance_df
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("RANDOM FOREST FEATURE IMPORTANCE")
print("--------------------------------")

print(rf_importance_df.to_string(index=False))

print("RANDOM FOREST IMPORTANCE COMPLETED")


In [ ]:
import pandas as pd
import numpy as np

lr_coefficients = lr_model.coefficients.toArray()

lr_coef_df = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": lr_coefficients,
    "absolute_coefficient": np.abs(lr_coefficients)
})

lr_coef_df = (
    lr_coef_df
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("LOGISTIC REGRESSION COEFFICIENTS")
print("--------------------------------")

print(
    lr_coef_df.to_string(
        index=False
    )
)

print("LOGISTIC REGRESSION EXPLAINABILITY COMPLETED")


In [ ]:
import matplotlib.pyplot as plt

plot_df = rf_importance_df.sort_values(
    "importance",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_df["feature"],
    plot_df["importance"]
)

plt.title(
    "Random Forest Feature Importance\n"
    "Synthetic Network Security Dataset"
)

plt.xlabel("Feature Importance")
plt.ylabel("Network Security Feature")

plt.tight_layout()
plt.show()

print("FEATURE IMPORTANCE VISUALIZATION COMPLETED")


In [ ]:
from pathlib import Path

output_dir = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "cyber_threat_detection/output"
)

output_dir.mkdir(parents=True, exist_ok=True)

rf_path = (
    output_dir / "random_forest_feature_importance.csv"
)

lr_path = (
    output_dir / "logistic_regression_coefficients.csv"
)

rf_importance_df.to_csv(
    rf_path,
    index=False
)

lr_coef_df.to_csv(
    lr_path,
    index=False
)

print("EXPLAINABILITY FILES SAVED")
print("--------------------------")
print("Random Forest:", rf_path)
print("Logistic Regression:", lr_path)

assert rf_path.exists()
assert lr_path.exists()

print("MODEL EXPLAINABILITY EXPORT COMPLETED")


In [ ]:
import numpy as np

importance_total = rf_importance_df[
    "importance"
].sum()

print("EXPLAINABILITY VALIDATION")
print("-------------------------")

print(
    "Random Forest importance sum:",
    round(importance_total, 6)
)

print(
    "Number of features:",
    len(rf_importance_df)
)

assert np.isclose(importance_total, 1.0)

assert len(rf_importance_df) == 8

assert len(lr_coef_df) == 8

print("EXPLAINABILITY VALIDATION PASSED")


## 10. Build report and run QA checks


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

project_base = Path(
    "/Volumes/workspace/usda_training/raw_files/"
    "cyber_threat_detection"
)

reports_dir = project_base / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / "MODEL_EVALUATION_REPORT.md"

top_features = "\n".join(
    f"- {row.feature}: {row.importance:.4f}"
    for row in rf_importance_df.itertuples(index=False)
)

report = f"""# Cybersecurity Threat Detection — Model Evaluation

## Project Overview
An educational machine-learning demonstration using
5,000 synthetically generated network security events.

## Dataset
- Total events: 5,000
- Normal events: 4,223
- Suspicious events: 777
- Training events: {train_count}
- Testing events: {test_count}
- Numeric model features: {len(feature_columns)}
- Test strategy: seeded random 80/20 split

## Logistic Regression
- Accuracy: {lr_results['accuracy']:.4f}
- Suspicious precision: {lr_security['precision']:.4f}
- Suspicious recall: {lr_security['recall']:.4f}
- Suspicious F1: {lr_security['f1']:.4f}
- True positives: {lr_security['TP']}
- True negatives: {lr_security['TN']}
- False positives: {lr_security['FP']}
- False negatives: {lr_security['FN']}

## Random Forest
- Accuracy: {rf_results['accuracy']:.4f}
- Suspicious precision: {rf_security['precision']:.4f}
- Suspicious recall: {rf_security['recall']:.4f}
- Suspicious F1: {rf_security['f1']:.4f}
- True positives: {rf_security['TP']}
- True negatives: {rf_security['TN']}
- False positives: {rf_security['FP']}
- False negatives: {rf_security['FN']}

## Random Forest Feature Importance
{top_features}

## Experiment Tracking
- Tracking platform: Databricks MLflow
- Logistic Regression run: {model_runs['Logistic_Regression']}
- Random Forest run: {model_runs['Random_Forest']}

## Limitations
1. All security events are synthetic.
2. Labels were assigned before feature generation.
3. Normal and suspicious events have deliberately
   separable feature patterns, inflating performance.
4. The random split does not establish performance
   against future or independently collected attacks.
5. Feature importance is not a causal explanation.
6. The models have not been validated for production
   intrusion detection or automated enforcement.

## Conclusion
Both Spark MLlib classifiers completed training and
held-out evaluation. Random Forest achieved perfect
classification on this synthetic test split, but
real-world effectiveness remains unverified.
"""

report_path.write_text(report, encoding="utf-8")

print("TECHNICAL REPORT CREATED")
print("Path:", report_path)
print("Size:", report_path.stat().st_size, "bytes")


In [ ]:
import numpy as np
import mlflow

checks = {
    "5,000 records processed":
        security_df.count() == 5000,

    "Training and testing records reconcile":
        train_count + test_count == 5000,

    "All eight model features present":
        len(feature_columns) == 8,

    "Random Forest importance sums to one":
        np.isclose(
            rf_importance_df["importance"].sum(), 1.0
        ),

    "Logistic Regression evaluated 958 events":
        sum(lr_security[k] for k in
            ["TP", "TN", "FP", "FN"]) == 958,

    "Random Forest evaluated 958 events":
        sum(rf_security[k] for k in
            ["TP", "TN", "FP", "FN"]) == 958,

    "Random Forest detected 138 synthetic positives":
        rf_security["TP"] == 138,

    "Logistic Regression missed one positive":
        lr_security["FN"] == 1,

    "Feature importance CSV exists":
        rf_path.exists(),

    "Regression coefficients CSV exists":
        lr_path.exists(),

    "Technical report exists":
        report_path.exists(),

    "Both MLflow runs finished":
        all(
            mlflow.get_run(run_id).info.status == "FINISHED"
            for run_id in model_runs.values()
        )
}

print("PROJECT 4 FINAL QUALITY ASSURANCE")
print("---------------------------------")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values()), (
    "One or more Project 4 QA checks failed"
)

print("---------------------------------")
print("ALL PROJECT 4 QUALITY CHECKS PASSED")
